# MuscleMap WB + SLM-SAM2

Uses [SLM-SAM2](https://github.com/mazurowski-lab/SLM-SAM2) (Short-Long Memory SAM2)
to propagate MuscleMap masks through the full 3D volume from a single prompt slice.

**Pipeline per stack:**
1. Export NIfTI slices as JPEG files to a temp folder
2. Choose a prompt slice (the one with maximum total muscle coverage)
3. `init_state` — compute SAM2 image embeddings for all slices at once
4. `add_new_mask` — add the MuscleMap mask for each muscle as a separate `obj_id`
5. `propagate_in_video` forward then backward from the prompt slice
6. Collect per-slice masks and save as npz

**Requires:** SLM-SAM2 installed (`pip install -e .` in its repo root) and
SAM 2.1 weights downloaded via `checkpoints/download_ckpts.sh`.

Output folder: `MuscleMap_WB_slmsam2/`

**Kernel:** `dafne_clean` (or any env with SLM-SAM2 installed)

In [1]:
import glob
import os
import sys
import tempfile
import numpy as np
import SimpleITK as sitk
import torch
from PIL import Image

# --- point to your local SLM-SAM2 clone ---
SLMSAM2_DIR = r"C:\Projects\SLM-SAM2"   # adjust if cloned elsewhere
if SLMSAM2_DIR not in sys.path:
    sys.path.insert(0, SLMSAM2_DIR)

# must run from the repo root so Hydra can find the configs
os.chdir(SLMSAM2_DIR)

from sam2.build_sam import build_sam2_video_predictor

In [2]:
# --- SAM2 checkpoint (download via checkpoints/download_ckpts.sh) ---
CHECKPOINT = os.path.join(SLMSAM2_DIR, "checkpoints", "sam2.1_hiera_base_plus.pt")
MODEL_CFG  = "configs/sam2.1/sam2.1_hiera_b+.yaml"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

# --- data paths (relative to eval_notebooks) ---
EVAL_DIR    = r"C:\Projects\dissector\eval_notebooks"
MM_SEGS_DIR = os.path.join(EVAL_DIR, "MuscleMap_segs")
IMAGE_GLOB  = os.path.join(EVAL_DIR, "myosegmenTUM", "*", "ImageData",
                           "*FATFRACTION", "*FATFRACTION_stack*.nii")
OUTPUT_DIR  = os.path.join(EVAL_DIR, "MuscleMap_WB_slmsam2")

os.makedirs(OUTPUT_DIR, exist_ok=True)

LABEL_MAP = {
    7101: "Vastus_Lateralis_L",
    7102: "Vastus_Lateralis_R",
    7111: "Vastus_Intermedius_L",
    7112: "Vastus_Intermedius_R",
    7121: "Vastus_Medialis_L",
    7122: "Vastus_Medialis_R",
    7131: "Rectus_Femoris_L",
    7132: "Rectus_Femoris_R",
    7141: "Sartorius_L",
    7142: "Sartorius_R",
    7151: "Gracilis_L",
    7152: "Gracilis_R",
    7161: "Semimembranosus_L",
    7162: "Semimembranosus_R",
    7171: "Semitendinosus_L",
    7172: "Semitendinosus_R",
    7181: "Biceps_Femoris_L",
    7182: "Biceps_Femoris_R",
    7201: "Adductor_Magnus_L",
    7202: "Adductor_Magnus_R",
}
print(f"{len(LABEL_MAP)} muscle labels")

Device: cpu
20 muscle labels


In [4]:
predictor = build_sam2_video_predictor(MODEL_CFG, CHECKPOINT, device=DEVICE)
predictor.eval()
print("SLM-SAM2 loaded on", DEVICE)

ERROR:root:['rec_maskmem_tpos_enc']


SLM-SAM2 loaded on cpu


In [5]:
def export_slices_to_jpeg(img_array, out_dir):
    """
    Save each slice of a (slices, H, W) float array as a JPEG in out_dir.
    Files are named 0.jpg, 1.jpg, ... as SAM2 expects.
    """
    vmin, vmax = img_array.min(), img_array.max()
    for i, slc in enumerate(img_array):
        slc_uint8 = ((slc - vmin) / (vmax - vmin + 1e-8) * 255).astype(np.uint8)
        rgb = np.stack([slc_uint8] * 3, axis=-1)   # SAM2 expects RGB
        Image.fromarray(rgb).save(os.path.join(out_dir, f"{i}.jpg"))


def best_prompt_frame(seg_array, label_map):
    """
    Return the slice index with the largest total labelled area across all
    muscles in label_map — this is the most informative prompt frame.
    """
    totals = np.zeros(seg_array.shape[0], dtype=np.int64)
    for label_idx in label_map:
        totals += (seg_array == label_idx).sum(axis=(1, 2))
    return int(totals.argmax())

In [6]:
image_files = sorted(glob.glob(IMAGE_GLOB))
matched, missing = [], []
for nii_path in image_files:
    stem = os.path.splitext(os.path.basename(nii_path))[0]
    seg_path = os.path.join(MM_SEGS_DIR, f"{stem}_dseg.nii.gz")
    if os.path.exists(seg_path):
        matched.append(nii_path)
    else:
        missing.append(stem)
print(f"{len(matched)} stacks with MuscleMap seg, {len(missing)} without")

54 stacks with MuscleMap seg, 0 without


In [ ]:
for nii_path in matched:
    stem     = os.path.splitext(os.path.basename(nii_path))[0]
    out_path = os.path.join(OUTPUT_DIR, f"{stem}_mm_slmsam2.npz")

    if os.path.exists(out_path):
        print(f"Skipping (already done): {out_path}")
        continue

    seg_path = os.path.join(MM_SEGS_DIR, f"{stem}_dseg.nii.gz")
    print(f"\nProcessing: {nii_path}")

    img_sitk  = sitk.ReadImage(nii_path)
    img_array = sitk.GetArrayFromImage(img_sitk).astype(float)   # (slices, H, W)
    seg_sitk  = sitk.ReadImage(seg_path)
    seg_array = sitk.GetArrayFromImage(seg_sitk)                  # (slices, H, W), int labels
    n_slices  = img_array.shape[0]
    print(f"  Shape: {img_array.shape}")

    # which muscles are actually present in this stack?
    present = {k: v for k, v in LABEL_MAP.items() if np.any(seg_array == k)}
    if not present:
        print("  No recognised labels found — skipping")
        continue

    prompt_frame = best_prompt_frame(seg_array, present)
    print(f"  Prompt frame: {prompt_frame}  ({len(present)} muscles present)")

    with tempfile.TemporaryDirectory() as tmpdir:
        # 1. export slices as JPEG
        export_slices_to_jpeg(img_array, tmpdir)

        # 2. init SAM2 state — computes embeddings for all slices
        inference_state = predictor.init_state(video_path=tmpdir)

        # 3. add each muscle mask as a separate obj_id on the prompt frame
        obj_id_to_name = {}
        for obj_id, (label_idx, muscle_name) in enumerate(present.items(), start=1):
            mask_2d = (seg_array[prompt_frame] == label_idx).astype(np.uint8)
            predictor.add_new_mask(
                inference_state=inference_state,
                frame_idx=prompt_frame,
                obj_id=obj_id,
                mask=mask_2d,
            )
            obj_id_to_name[obj_id] = muscle_name

        # 4. propagate forward from prompt frame
        video_segments = {}
        for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
            inference_state, start_frame_idx=prompt_frame, recent_n=1
        ):
            video_segments[out_frame_idx] = {
                oid: (out_mask_logits[i] > 0).cpu().numpy()
                for i, oid in enumerate(out_obj_ids)
            }

        # 5. propagate backward if prompt is not the first slice
        if prompt_frame > 0:
            for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(
                inference_state, start_frame_idx=prompt_frame, reverse=True, recent_n=1
            ):
                video_segments[out_frame_idx] = {
                    oid: (out_mask_logits[i] > 0).cpu().numpy()
                    for i, oid in enumerate(out_obj_ids)
                }

    # 6. assemble 3D masks and save
    H, W = img_array.shape[1], img_array.shape[2]
    all_masks = {name: np.zeros((n_slices, H, W), dtype=np.uint8)
                 for name in obj_id_to_name.values()}

    for frame_idx, obj_masks in video_segments.items():
        for obj_id, mask in obj_masks.items():
            muscle_name = obj_id_to_name[obj_id]
            all_masks[muscle_name][frame_idx] = np.squeeze(mask).astype(np.uint8)

    np.savez_compressed(out_path, **all_masks)
    print(f"  Saved -> {out_path}")

    # per-file summary
    mm_voxels = {LABEL_MAP[k]: int((seg_array == k).sum()) for k in present}
    print(f"  {'Muscle':<30} {'MM input':>10} {'SLM-SAM2':>10}")
    print(f"  {'-'*52}")
    for name, vol in sorted(all_masks.items()):
        refined_v = int(vol.sum())
        input_v   = mm_voxels.get(name, 0)
        flag      = "  <-- EMPTY" if refined_v == 0 and input_v > 0 else ""
        print(f"  {name:<30} {input_v:>10,} {refined_v:>10,}{flag}")

print("\nAll done.")


Processing: C:\Projects\dissector\eval_notebooks\myosegmenTUM\HV001_1\ImageData\HV001_1_FATFRACTION\HV001_1_FATFRACTION_stack1.nii
  Shape: (65, 672, 672)
  Prompt frame: 0  (19 muscles present)


frame loading (JPEG): 100%|███████████████████████████████████████████████████████████████████████████████| 65/65 [00:05<00:00, 12.52it/s]
C:\Projects\SLM-SAM2\sam2\sam2_video_predictor.py:981: UserWarning: cannot import name '_C' from 'sam2' (C:\Projects\SLM-SAM2\sam2\__init__.py)

Skipping the post-processing step due to the error above. You can still use SAM 2 and it's OK to ignore the error above, although some post-processing functionality may be limited (which doesn't affect the results in most cases; see https://github.com/facebookresearch/sam2/blob/main/INSTALL.md).
  pred_masks_gpu = fill_holes_in_mask_scores(


In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.npz")))
if results:
    sample = np.load(results[0])
    print("Sample:", results[0])
    for name in sample.files:
        arr = sample[name]
        print(f"  {name}: shape={arr.shape}  voxels={arr.sum()}")
else:
    print("No results yet.")